# YouTube Video Embedding with ChromaDB


이 노트북은 유튜브 영상 링크 리스트를 입력받아 영상 데이터(제목, 캡션 등)를 전처리하고, SentenceTransformer 기반 임베딩을 생성한 뒤 ChromaDB 컬렉션에 저장하는 예시를 제공합니다.

> ⚠️ **주의**: 실제 실행 시에는 네트워크 연결과 OpenAI API Key 등 외부 리소스 접근 권한이 필요할 수 있습니다. 네트워크 접근이 어려운 환경에서는 이 노트북을 그대로 실행할 수 없습니다.


In [ ]:
# 필수 라이브러리 설치
%pip install --quiet chromadb yt-dlp youtube-transcript-api sentence-transformers


In [ ]:
import os
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from yt_dlp import YoutubeDL
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, NoTranscriptAvailable


In [ ]:
# 유튜브 영상 URL 목록 입력
video_urls = ["https://www.youtube.com/watch?v=dQw4w9WgXcQ"]  # 필요에 맞게 수정하세요.

# SentenceTransformer 모델 이름 (필요에 따라 다른 모델을 사용할 수 있습니다.)
embedding_model_name = "all-MiniLM-L6-v2"

# ChromaDB 설정
persist_directory = Path("./chroma_store")
persist_directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def extract_video_id(url: str) -> str:
    """주어진 YouTube URL에서 비디오 ID를 추출합니다."""
    parsed = urlparse(url)
    if parsed.hostname in ("youtu.be", "www.youtu.be"):
        return parsed.path.lstrip('/')
    if parsed.hostname and 'youtube.com' in parsed.hostname:
        if parsed.path == '/watch':
            return parse_qs(parsed.query).get('v', [''])[0]
        if parsed.path.startswith('/shorts/'):
            return parsed.path.split('/')[-1]
        if parsed.path.startswith('/live/'):
            return parsed.path.split('/')[-1]
    raise ValueError(f'유효한 YouTube URL이 아닙니다: {url}')

def fetch_video_metadata(url: str) -> dict:
    """yt_dlp을 이용해 기본 메타데이터와 자동 캡션을 가져옵니다."""
    ydl_opts = {
        'quiet': True,
        'skip_download': True,
        'writesubtitles': False,
        'writeautomaticsub': False
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    video_id = info.get('id')
    title = info.get('title')
    channel = info.get('uploader')
    duration = info.get('duration')
    description = info.get('description') or ''
    automatic_captions = info.get('automatic_captions') or {}
    return {
        'video_id': video_id,
        'title': title,
        'channel': channel,
        'duration': duration,
        'description': description,
        'automatic_captions': automatic_captions
    }

def fetch_transcript_text(video_id: str, metadata: dict, languages=None) -> str:
    """YouTubeTranscriptApi를 이용해 캡션 텍스트를 수집합니다."""
    languages = languages or ['ko', 'en']
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        for language in languages:
            try:
                transcript = transcript_list.find_transcript([language])
                segments = transcript.fetch()
                return ' '.join(segment['text'] for segment in segments if segment.get('text'))
            except NoTranscriptFound:
                continue
    except (TranscriptsDisabled, NoTranscriptAvailable):
        pass

    # 자동 캡션 fallback
    auto_caps = metadata.get('automatic_captions') or {}
    for lang_code, entries in auto_caps.items():
        if not entries:
            continue
        if entries[0].get('ext') == 'vtt':
            # yt_dlp는 URL 리스트를 제공 -> 텍스트 추출은 직접 처리 필요
            # 간단한 예시로는 실제 다운로드를 생략하고 URL 정보만 표시합니다.
            return f'자동 캡션({lang_code})을 사용할 수 있습니다: {entries[0].get('url')}'
    return ''


In [ ]:
def build_documents(video_urls):
    documents = []
    metadatas = []
    ids = []

    for url in video_urls:
        metadata = fetch_video_metadata(url)
        video_id = metadata['video_id']
        captions_text = fetch_transcript_text(video_id, metadata)
        title_text = metadata['title'] or ''
        description = metadata['description'] or ''

        # video_data는 제목, 설명, 캡션을 모두 결합한 텍스트로 구성합니다.
        video_data_text = '

'.join(filter(None, [title_text, description, captions_text]))

        common_metadata = {
            'url': url,
            'video_id': video_id,
            'duration': metadata['duration'],
            'channel': metadata['channel']
        }

        payloads = [
            (f'{video_id}_video', video_data_text, {**common_metadata, 'field': 'video_data'}),
            (f'{video_id}_title', title_text, {**common_metadata, 'field': 'title'}),
            (f'{video_id}_captions', captions_text, {**common_metadata, 'field': 'captions'})
        ]

        for payload_id, document, metadata_payload in payloads:
            if not document:
                continue
            ids.append(payload_id)
            documents.append(document)
            metadatas.append(metadata_payload)

    return ids, documents, metadatas


In [ ]:
# 문서 및 메타데이터 구성
ids, documents, metadatas = build_documents(video_urls)
print(f'총 {len(documents)}개의 문서를 생성했습니다.')
if documents:
    print('샘플 ID:', ids[0])
    print('샘플 메타데이터:', metadatas[0])


In [ ]:
# SentenceTransformer Embedding Function 정의
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embedding_model_name)

client = chromadb.PersistentClient(
    path=str(persist_directory),
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_or_create_collection(
    name='youtube_video_embeddings',
    metadata={"description": "YouTube 영상 임베딩"},
    embedding_function=embedding_function
)

if documents:
    collection.add(ids=ids, metadatas=metadatas, documents=documents)
    print(f'컬렉션에 {len(documents)}개의 항목을 추가했습니다.')
else:
    print('추가할 문서가 없습니다.')


In [ ]:
# 간단한 유사도 검색 예시
if documents:
    query_text = '흥겨운 80년대 팝송'
    results = collection.query(query_texts=[query_text], n_results=3)
    for doc_id, doc, metadata in zip(results['ids'][0], results['documents'][0], results['metadatas'][0]):
        print('-' * 80)
        print('ID:', doc_id)
        print('URL:', metadata.get('url'))
        print('필드:', metadata.get('field'))
        print('채널:', metadata.get('channel'))
        print('영상 길이(초):', metadata.get('duration'))
        print('문서 일부:', doc[:200], '...')
else:
    print('컬렉션에 문서가 없으므로 검색을 수행할 수 없습니다.')
